# Model Evaluation - Multi-Crop Leaf Disease Detection

This notebook evaluates trained models on the test dataset and generates performance metrics including:
- Overall accuracy and F1-scores
- Per-class precision, recall, and F1-scores
- Confusion matrix visualization
- Performance analysis report

**Prerequisites:**
- Trained model (.h5 file)
- Test dataset in `/processed/test/` with class subdirectories
- Configuration file with model paths

## Step 1: Import Required Libraries

In [ ]:
import os
import yaml
import numpy as np
import pandas as pd
from tensorflow import keras
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## Step 2: Configure Paths and Load Configuration

In [ ]:
# Configuration for evaluation
# Modify these paths based on your setup

# LOCAL SETUP (on your computer)
MODEL_PATH = "../models/mobilenetv2/mobilenetv2_best.h5"  # Path to trained model
CONFIG_PATH = "../training/config_mobilenetv2.yaml"  # Path to config
TEST_DIR = "../dataset/processed/test"  # Path to test dataset

# COLAB SETUP (uncomment if using Colab)
# MODEL_PATH = "/content/drive/MyDrive/leaf_models/mobilenetv2/mobilenetv2_best.h5"
# CONFIG_PATH = "/content/multi-crop-leaf-disease-detection/training/config_mobilenetv2.yaml"
# TEST_DIR = "/content/processed/test"

# Load configuration
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

print(f"✓ Configuration loaded from {CONFIG_PATH}")
print(f"✓ Model: {MODEL_PATH}")
print(f"✓ Test dataset: {TEST_DIR}")

## Step 3: Load Trained Model

In [ ]:
print(f"Loading model from {MODEL_PATH}...")
model = keras.models.load_model(MODEL_PATH)
print(f"✓ Model loaded successfully!")
print(f"\nModel Summary:")
model.summary()

## Step 4: Load Test Dataset

In [ ]:
print(f"Loading test dataset from {TEST_DIR}...")

batch_size = config.get("dataset", {}).get("batch_size", 32)
image_size = tuple(config.get("model", {}).get("input_shape", [224, 224])[:2])

test_ds = keras.preprocessing.image_dataset_from_directory(
    TEST_DIR,
    image_size=image_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=False,
)

class_names = test_ds.class_names
print(f"✓ Test dataset loaded!")
print(f"✓ Found {len(class_names)} classes:")
for i, cls in enumerate(class_names, 1):
    print(f"  {i}. {cls}")

## Step 5: Generate Predictions and Compute Metrics

In [ ]:
print("Generating predictions...")
y_pred = []
y_true = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(predictions, axis=1))
    y_true.extend(np.argmax(labels.numpy(), axis=1))

y_pred = np.array(y_pred)
y_true = np.array(y_true)

print(f"✓ Predictions generated for {len(y_true)} test samples\n")

# Compute overall accuracy
accuracy = accuracy_score(y_true, y_pred)

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, average=None, labels=range(len(class_names))
)

# Macro and weighted averages
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro"
)
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted"
)

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"\n🎯 Overall Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)\n")

print("📊 Macro Average (unweighted):")
print(f"   Precision: {precision_macro:.4f}")
print(f"   Recall:    {recall_macro:.4f}")
print(f"   F1-Score:  {f1_macro:.4f}")

print("\n⚖️  Weighted Average (by class frequency):")
print(f"   Precision: {precision_weighted:.4f}")
print(f"   Recall:    {recall_weighted:.4f}")
print(f"   F1-Score:  {f1_weighted:.4f}")

## Step 6: Classification Report

In [ ]:
print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (by class)")
print("=" * 60 + "\n")
print(classification_report(y_true, y_pred, target_names=class_names))

## Step 7: Per-Class Performance Table

In [ ]:
# Create DataFrame for better visualization
metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'Support': support
})

# Sort by F1-Score
metrics_df = metrics_df.sort_values('F1-Score', ascending=False)

print("\n📋 Per-Class Metrics (sorted by F1-Score):")
print(metrics_df.to_string(index=False))

# Save to CSV
os.makedirs("../results", exist_ok=True)
metrics_df.to_csv("../results/evaluation_metrics.csv", index=False)
print("\n✓ Metrics saved to ../results/evaluation_metrics.csv")

## Step 8: Confusion Matrix Visualization

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Normalize confusion matrix
row_sums = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm.astype('float'),
    row_sums,
    out=np.zeros_like(cm, dtype=float),
    where=row_sums != 0,
)

# Plot normalized confusion matrix
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Normalized Count'},
)
plt.title('Confusion Matrix (Normalized)', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=14)
plt.xlabel('Predicted Label', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=300, bbox_inches='tight')
print("✓ Confusion matrix saved to ../results/confusion_matrix.png")
plt.show()

## Step 9: Performance Analysis

In [ ]:
# Best performing classes (top 5 by F1-Score)
print("\n" + "=" * 60)
print("🏆 TOP 5 BEST PERFORMING CLASSES (by F1-Score)")
print("=" * 60)
top_5_idx = np.argsort(f1)[-5:][::-1]
for rank, idx in enumerate(top_5_idx, 1):
    print(f"{rank}. {class_names[idx]:<40} F1={f1[idx]:.4f}")

# Worst performing classes (bottom 5 by F1-Score)
print("\n" + "=" * 60)
print("⚠️  BOTTOM 5 WORST PERFORMING CLASSES (by F1-Score)")
print("=" * 60)
bottom_5_idx = np.argsort(f1)[:5]
for rank, idx in enumerate(bottom_5_idx, 1):
    print(f"{rank}. {class_names[idx]:<40} F1={f1[idx]:.4f}")

# Visualization: F1-Scores by class
fig, ax = plt.subplots(figsize=(14, 8))
sorted_idx = np.argsort(f1)
sorted_classes = [class_names[i] for i in sorted_idx]
sorted_f1 = f1[sorted_idx]

colors = ['red' if score < 0.8 else 'orange' if score < 0.9 else 'green' for score in sorted_f1]
ax.barh(sorted_classes, sorted_f1, color=colors)
ax.axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Warning threshold (0.8)')
ax.axvline(x=0.9, color='orange', linestyle='--', linewidth=2, label='Good threshold (0.9)')
ax.set_xlabel('F1-Score', fontsize=12)
ax.set_title('F1-Scores by Class', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1])
ax.legend()
plt.tight_layout()
plt.savefig('../results/f1_scores_by_class.png', dpi=300, bbox_inches='tight')
print("\n✓ F1-Score chart saved to ../results/f1_scores_by_class.png")
plt.show()

## Step 10: Summary Report

In [ ]:
# Generate markdown report
report_path = "../results/evaluation_report.md"

with open(report_path, "w") as f:
    f.write("# Model Evaluation Report\n\n")
    f.write(f"**Model**: {MODEL_PATH}\n")
    f.write(f"**Test Dataset**: {TEST_DIR}\n")
    f.write(f"**Test Samples**: {len(y_true)}\n\n")
    
    f.write("## Overall Performance\n\n")
    f.write(f"- **Accuracy**: {accuracy:.4f} ({accuracy * 100:.2f}%)\n")
    f.write(f"- **Precision (macro)**: {precision_macro:.4f}\n")
    f.write(f"- **Recall (macro)**: {recall_macro:.4f}\n")
    f.write(f"- **F1-Score (macro)**: {f1_macro:.4f}\n")
    f.write(f"- **Precision (weighted)**: {precision_weighted:.4f}\n")
    f.write(f"- **Recall (weighted)**: {recall_weighted:.4f}\n")
    f.write(f"- **F1-Score (weighted)**: {f1_weighted:.4f}\n\n")
    
    f.write("## Per-Class Performance\n\n")
    f.write("| Class | Precision | Recall | F1-Score | Support |\n")
    f.write("|-------|-----------|--------|----------|----------|\n")
    for i, name in enumerate(class_names):
        f.write(f"| {name} | {precision[i]:.4f} | {recall[i]:.4f} | {f1[i]:.4f} | {int(support[i])} |\n")
    
    f.write("\n## Key Insights\n\n")
    
    # Classes with F1 < 0.8
    poor_classes = [class_names[i] for i in range(len(class_names)) if f1[i] < 0.8]
    if poor_classes:
        f.write("### Classes Needing Improvement (F1 < 0.8)\n")
        for cls in poor_classes:
            idx = class_names.index(cls)
            f.write(f"- **{cls}**: F1={f1[idx]:.4f} (Recommendations: collect more data, increase augmentation)\n")
        f.write("\n")
    
    # Statistics
    f.write("### Statistics\n")
    f.write(f"- Total test samples: {len(y_true)}\n")
    f.write(f"- Correctly classified: {np.sum(y_pred == y_true)}\n")
    f.write(f"- Misclassified: {np.sum(y_pred != y_true)}\n")
    f.write(f"- Classes: {len(class_names)}\n")
    f.write(f"- Best F1-Score: {np.max(f1):.4f} ({class_names[np.argmax(f1)]})\n")
    f.write(f"- Worst F1-Score: {np.min(f1):.4f} ({class_names[np.argmin(f1)]})\n")

print(f"\n✓ Evaluation report saved to {report_path}")

# Display summary
print("\n" + "=" * 60)
print("📊 EVALUATION SUMMARY")
print("=" * 60)
print(f"\nAccuracy: {accuracy * 100:.2f}%")
print(f"F1-Score (macro): {f1_macro:.4f}")
print(f"F1-Score (weighted): {f1_weighted:.4f}")
print(f"\nTest Samples: {len(y_true)}")
print(f"Correct: {np.sum(y_pred == y_true)} | Incorrect: {np.sum(y_pred != y_true)}")
print(f"\n✅ All evaluation results saved to ../results/")
print("   - evaluation_metrics.csv")
print("   - confusion_matrix.png")
print("   - f1_scores_by_class.png")
print("   - evaluation_report.md")

## How to Use This Notebook

### For Different Models

**Evaluate MobileNetV2:**
```python
MODEL_PATH = "../models/mobilenetv2/mobilenetv2_best.h5"
CONFIG_PATH = "../training/config_mobilenetv2.yaml"
```

**Evaluate EfficientNet-Lite0:**
```python
MODEL_PATH = "../models/efficientnet_lite0/efficientnet_lite0_best.h5"
CONFIG_PATH = "../training/config_efficientnet_lite0.yaml"
```

### For Different Datasets

**Local test set:**
```python
TEST_DIR = "../dataset/processed/test"
```

**Colab:**
```python
TEST_DIR = "/content/processed/test"
```

### Output Files

The notebook generates:
- **evaluation_metrics.csv** — Per-class metrics (precision, recall, F1)
- **confusion_matrix.png** — Normalized confusion matrix heatmap
- **f1_scores_by_class.png** — Bar chart of F1-scores
- **evaluation_report.md** — Complete markdown report

### Interpreting Results

- **Accuracy > 90%**: Excellent performance
- **Accuracy 80-90%**: Good, room for improvement
- **Accuracy < 80%**: Needs more training data or model tuning

Classes with F1 < 0.8 should be investigated for:
- More training samples needed
- Data augmentation required
- Potential class imbalance issues